In [1]:
import numpy as np
import pandas as pd
import pickle

# Load from Modelling output
df2 = pd.read_csv('data_preprocessed.csv')
df2['capturedAt'] = pd.to_datetime(df2['capturedAt'])
df2['date'] = df2['capturedAt'].dt.date
with open('feature_cols.pkl', 'rb') as f:
    obj = pickle.load(f)
global_median = obj['global_median']
with open('model.pkl', 'rb') as f:
    obj2 = pickle.load(f)
model = obj2['model']
print('✓ All artifacts loaded')

✓ All artifacts loaded


# Test File Prediction
Using the built pipeline to predict prices in the 3-day test file (March 22–24, 2025).

Daily workflow:
1. Separate the anchor (100 rows with known prices)
2. Calculate the ratio from that day's anchor
3. Predict empty rows using Tier 2: last_price × ratio
4. Save results to CSV


## 1. Loading and Preprocessing the Test File

In [2]:
test = pd.read_csv("ecommerce_price_prediction-test-3-days.csv")
test['capturedAt'] = pd.to_datetime(test['capturedAt'])
test['date']       = test['capturedAt'].dt.date

print(f"Total rows: {len(test):,}")
for date in sorted(test['date'].unique()):
    day          = test[test['date'] == date]
    available    = day['price'].notna().sum()
    empty        = day['price'].isna().sum()
    print(f"  {date} → {available} anchors, {empty:,} to be predicted")

Total rows: 25,900
  2025-03-22 → 100 anchors, 9,214 to be predicted
  2025-03-23 → 100 anchors, 7,953 to be predicted
  2025-03-24 → 100 anchors, 8,433 to be predicted


Each day has 100 anchors and thousands of empty rows. The anchors vary from day to day, so we process day by day to ensure the calibration ratio for each day is calculated from that specific day's anchors.

## 2. Merge Historical Features to Test
Historical features are extracted from the ENTIRE df2 (not just the train set) because when predicting the test set, we can use all available training data, including March 22, which we previously used for validation. The more recent the last known price, the more accurate it will be.

In [3]:
hist_all = (
    df2.groupby('modelId')['price']
    .agg(
        model_last_price='last',
        model_median_price='median',
        model_mean_price='mean',
        model_count='count'
    )
    .reset_index()
)

test = test.merge(hist_all, on='modelId', how='left')
test['model_last_price']   = test['model_last_price'].fillna(global_median)
test['model_median_price'] = test['model_median_price'].fillna(global_median)
test['model_mean_price']   = test['model_mean_price'].fillna(global_median)
test['model_count']        = test['model_count'].fillna(0)

print("Historical features successfully merged")
print(f"Test products not present in training: {(test['model_count']==0).sum()}")

Historical features successfully merged
Test products not present in training: 2


Almost all test products are present in the training data. Those that are not present (cold start) are filled with global_median as a fallback.

## 3. Daily Prediction
Each day is processed separately because the anchor ratio can differ each day, depending on the marketplace conditions on that day.

In [4]:
result = test.copy()
result['price_predicted'] = np.nan

for date in sorted(test['date'].unique()):
    # Get all rows for today
    mask_day       = result['date'] == date
    df_day         = result[mask_day]

    # Separate anchors vs rows to be predicted
    anchor_day     = df_day[df_day['price'].notna()]
    non_anchor_day = df_day[df_day['price'].isna()]

    # Calculate the ratio from today's anchors
    # Median is more robust to outliers than the mean
    day_ratio = np.median(
        anchor_day['price'].values /
        anchor_day['model_last_price'].values
    )

    # Tier 2 Prediction: last price × today's ratio
    predict = non_anchor_day['model_last_price'] * day_ratio
    result.loc[non_anchor_day.index, 'price_predicted'] = predict.values

    print(f"{date} → ratio={day_ratio:.4f}, "
          f"predict {len(non_anchor_day):,} rows")

# Fill empty price columns with predictions
result['price'] = result['price'].combine_first(result['price_predicted'])
result['price'] = result['price'].round(0)

print(f"\nTotal filled: {result['price'].notna().sum():,} / {len(result):,}")

2025-03-22 → ratio=1.0000, predict 9,214 rows
2025-03-23 → ratio=1.0000, predict 7,953 rows
2025-03-24 → ratio=1.0000, predict 8,433 rows

Total filled: 25,900 / 25,900


Each day's ratio shows the price shift for that day. A ratio of 1.0 means prices are stable. A ratio > 1 means prices have increased from the historical baseline. A ratio < 1 means prices have decreased.

## 4. Save Results to CSV

In [5]:
kolom_asli = ['capturedAt', 'shopId', 'itemId', 'modelId', 'price',
              'priceBeforeDiscount', 'promotionId', 'cat_id', 'stock',
              'normal_stock', 'raw_discount', 'show_discount', 'brand',
              'is_free_shipping', 'is_pre_order', 'item_price_min',
              'item_price_max', 'review_rating', 'total_rating_count',
              'cmt_count', 'shop_rating', 'shop_response_rate',
              'shop_follower_count', 'is_official_shop', 'is_verified',
              'is_preferred_plus_seller']

result[kolom_asli].to_csv("predictions_3days.csv", index=False)

print("✓ File saved: predictions_3days.csv")
print(f"\nSample prediction results:")
print(result[['capturedAt', 'modelId', 'price']].head(10).to_string())

✓ File saved: predictions_3days.csv

Sample prediction results:
               capturedAt       modelId       price
0 2025-03-22 04:27:05.325  139002028392   7500000.0
1 2025-03-22 04:27:05.325  139002028392   7500000.0
2 2025-03-22 04:27:05.325  241162534603   8400000.0
3 2025-03-22 04:27:05.325  241162534603   8400000.0
4 2025-03-22 04:27:05.325  241162534604   7500000.0
5 2025-03-22 04:27:05.325  241162534604   7500000.0
6 2025-03-22 04:27:05.325  241162534605   7500000.0
7 2025-03-22 04:27:05.325  241162534605   7500000.0
8 2025-03-22 04:27:05.388  194654575543  27800000.0
9 2025-03-22 04:27:05.388  226663185840  49500000.0
